# Granite 4.1 — Basic Usage

## Imports

In [2]:
from pprint import pprint

import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("MPS (Apple Silicon GPU) available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

torch: 2.13.0
transformers: 5.14.1
MPS (Apple Silicon GPU) available: True
CUDA available: False


## Load Model and Tokenizer

In [3]:
MODEL_ID = "ibm-granite/granite-4.1-3b"

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

print(f"Architecture: {model.config.architectures}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Architecture: ['GraniteForCausalLM']
Parameters: 3,402,836,480
Device: mps:0
Dtype: torch.bfloat16


## Single Turn Generation

In [12]:
messages = [
    {"role": "user", "content": "What is the capital of France?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|start_of_role|>user<|end_of_role|>What is the capital of France?<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>


In [13]:
inputs = tokenizer(chat, return_tensors='pt').to(model.device)
pprint(inputs, sort_dicts=False, width=120)

{'input_ids': tensor([[100264,    882, 100265,   3923,    374,    279,   6864,    315,   9822,
             30, 100257,    198, 100264,  78191, 100265]], device='mps:0'),
 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}


In [14]:
outputs = model.generate(**inputs, max_new_tokens=128)
print(outputs)

tensor([[100264,    882, 100265,   3923,    374,    279,   6864,    315,   9822,
             30, 100257,    198, 100264,  78191, 100265,    791,   6864,    315,
           9822,    374,  12366,     13, 100257]], device='mps:0')


In [15]:
print(tokenizer.decode(outputs[0]))

<|start_of_role|>user<|end_of_role|>What is the capital of France?<|end_of_text|>
<|start_of_role|>assistant<|end_of_role|>The capital of France is Paris.<|end_of_text|>


In [16]:
input_len = inputs["input_ids"].shape[-1]
response = tokenizer.decode(outputs[0][input_len:])
print(response)

The capital of France is Paris.<|end_of_text|>


In [17]:
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(response)

The capital of France is Paris.


## System Prompt

In [18]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who responds in all capitals."},
    {"role": "user", "content": "What is the capital of France?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

THE CAPITAL OF FRANCE IS PARIS.


## Multi-Turn Generation

In [19]:
messages = [
    {"role": "user", "content": "What's the capital of France?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

The capital of France is Paris.


In [20]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of France is Paris.'}]


In [21]:
prompt = "What is a famous landmark there?"

messages.append({"role": "user", "content": prompt})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the capital of France?"},
 {'role': 'assistant', 'content': 'The capital of France is Paris.'},
 {'role': 'user', 'content': 'What is a famous landmark there?'}]


In [22]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

A famous landmark in Paris is the Eiffel Tower.


## Streaming Generation

In [23]:
messages = [
    {"role": "user", "content": "What is a capital of France?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)

streamer = transformers.TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
outputs = model.generate(**inputs, max_new_tokens=128, streamer=streamer)

The capital of France is Paris.
